# Compliance Streaming — Evidence Notebook

## Purpose (Read-only Evidence)
This notebook exists **only to prove claims** about the Compliance Streaming system using **already-generated artifacts**.

It does **not**:
- run pipelines or watchers
- regenerate events
- recompute metrics beyond simple read-only counts
- modify any files

Each evidence “exhibit” is always written as **3 consecutive cells**:

1) **Explanation (Markdown)**  
- What claim is being proven  
- Which artifact(s) will be inspected  
- What would invalidate the claim

2) **Execution (Read-only Code)**  
- Only `print()`, file reads, JSON parsing, line counts, previews  
- No mutation, no re-runs, no “fixing”, no parameter changes

3) **Evidence (Markdown)**  
- What the displayed output proves (pass/fail)  
- No narrative beyond what is shown

---

## Declared Guarantees Under Review
This notebook evaluates evidence for:

- Persisted events in the append-only stream are **not skipped**
- Contract violations **fail-fast** with **no coercion**
- Rule evaluation is **deterministic** and window-based on `event_time`
- Restarts resume from the **last checkpointed byte offset**

---

## Explicit Non-Support (Boundaries)
The following are explicitly not supported and are not evaluated:

- Events not yet written to disk (pre-persistence loss boundary)
- Cross-process or cross-run deduplication (dedupe is process-scoped only)
- Exactly-once beyond a single watcher process boundary


## Section 1 — Run Log

### Explanation

This section inspects the **run log** to establish the **execution context** of the watcher.

The claim being proven:

- A watcher run occurred and produced an authoritative run record
- The run log captures immutable execution metadata (run identifier, timing, configuration context)

Artifact under inspection:

- `logs/run_<run_id>.json`

What this section **proves**:
- That this evidence notebook refers to a real watcher execution
- The temporal boundaries of that execution

What this section **does not prove**:
- Restart correctness
- Event completeness
- Rule correctness or outcomes

Those are proven in later sections using other artifacts.


In [40]:
# Section 1 — Run Log

import json
import pandas as pd

with open("../logs/run_success.json") as f:
    run_log = json.load(f)

pd.DataFrame.from_dict(run_log, orient="index", columns=["value"])


,value
run_id,b1a0816045df4208a3d14ed290490119
started_at,2026-03-09T12:40:32.416461+00:00
ended_at,2026-03-09T12:41:32.736893+00:00
status,success
contract,"{'name': 'money_flow_event_contract', 'version..."
paths,"{'stream': 'data/stream.log', 'flags': 'output..."
offsets,"{'resume_offset': 0, 'final_offset': 1870846}"
counts,"{'events_read': 6924, 'events_valid': 6924, 'f..."


### Evidence

The run log confirms that a **single watcher execution** occurred and completed successfully.

**Proves**
- A watcher run existed (`run_id` present)
- The run has clear temporal boundaries (`started_at`, `ended_at`)
- The stream and output artifact paths used in this run
- The contract name and version applied
- Final processing status (`success`)
- Aggregate counts reported by the watcher at shutdown

**Does NOT prove**
- Restart correctness
- Offset persistence across restarts
- Event completeness or ordering
- Rule correctness or flag validity

Those claims are evaluated in subsequent sections using checkpoints, flags, and validation artifacts.


## Section 2 — Stream Log (Persisted Input)

### Explanation

This section inspects the **persisted event stream** to establish the **input boundary** for the watcher.

The claim being proven:

- Events written to the append-only stream file form the authoritative input set
- The watcher processes only events that exist in this persisted file

Artifact under inspection:

- `data/stream.log`

What this section **proves**:
- That a persisted input stream exists
- The physical size and structure of the input boundary

What this section **does not prove**:
- That all events were processed
- Ordering or duplication guarantees
- Rule evaluation correctness

Those are proven using checkpoints and validation artifacts.


In [41]:
# Section 2 — Stream Log

import pandas as pd

stream_path = "../data/sample_stream.log"

# Read a small, fixed preview for inspection only
df = pd.read_json(stream_path, lines=True, nrows=5)

df


,event_id,user_id,event_time,ingest_time,event_type,amount,currency,channel
0,63277744-81b5-4568-8ead-b9ef97ca1839,user_05736,2026-03-09 12:36:24.201822+00:00,2026-03-09 12:36:24.201822+00:00,deposit_completed,63.61,GBP,api
1,3f664cc4-a363-4514-bd60-4ff992f63dfe,user_08206,2026-03-09 12:36:24.212288+00:00,2026-03-09 12:36:24.212288+00:00,deposit_completed,98.91,GBP,web
2,3414ab5b-4e9d-4331-a865-f25269333466,user_08723,2026-03-09 12:36:24.222452+00:00,2026-03-09 12:36:24.222452+00:00,deposit_completed,5.76,GBP,web
3,46e2b614-8156-41c0-9dfa-79ebe736950b,user_05093,2026-03-09 12:36:24.232625+00:00,2026-03-09 12:36:24.232625+00:00,deposit_completed,140.41,GBP,mobile
4,badad627-2ee7-4b9a-9b77-ed0388030e0c,user_09841,2026-03-09 12:36:24.242799+00:00,2026-03-09 12:36:24.242799+00:00,deposit_completed,45.32,GBP,mobile


### Evidence

The displayed rows confirm that a **persisted append-only stream file** exists and contains structured JSON event records.

**Proves**
- `data/sample_stream.log` exists and is readable
- Events are stored as line-delimited JSON records
- The watcher’s input is based on persisted data, not in-memory events

**Does NOT prove**
- That all persisted events were processed
- Event ordering or duplicate handling
- Rule evaluation or flag generation
- Restart correctness

Those claims are evaluated using checkpoint and validation artifacts.


## Section 3 — Checkpoint

### Explanation

This section inspects the **checkpoint artifact** to prove that the watcher
persisted its **last processed byte offset** to disk.

The claim being proven:

- A checkpoint file exists
- The checkpoint records a concrete byte `offset`
- The offset represents the persisted restart position at shutdown

Artifact under inspection:

- `outputs/checkpoints/state_success.json`

What this section **proves**:
- Restart position is explicitly persisted
- Restart safety exists within the persisted-stream boundary

What this section **does not prove**:
- Cross-run or cross-process exactly-once guarantees
- That no events were lost before persistence
- Rule correctness or validation outcomes


In [42]:
# Section 3 — Checkpoint

import json
import pandas as pd

with open("../outputs/checkpoints/state_success.json") as f:
    checkpoint = json.load(f)

pd.DataFrame.from_dict(checkpoint, orient="index", columns=["value"])


,value
offset,2287784
saved_at,2026-03-09T12:42:23.871674+00:00


### Evidence

The checkpoint artifact confirms that the watcher persisted its
**restart position** at shutdown.

**Proves**
- A checkpoint file exists
- A concrete byte `offset` is recorded
- A `saved_at` timestamp shows when the checkpoint was written

**Does NOT prove**
- That all persisted events were processed
- That restarts are idempotent across multiple processes
- That flags or validations were correct

Those claims are evaluated using flags and validation artifacts.


## Section 4 — Flags

### Explanation

This section inspects the **flags artifact** to show the **observable outputs**
of rule evaluation.

The claim being proven:

- Rule breaches, when detected, are emitted as persisted flag records
- Each flag records the rule name, triggering metric, and event context

Artifact under inspection:

- `outputs/flags/flags.log`

What this section **proves**:
- Rule outputs are materialised and persisted
- Flags contain sufficient context for audit and review

What this section **does not prove**:
- Correctness of rule logic
- Completeness of rule coverage
- Absence of missed breaches

Those are evaluated via validation summaries, not individual flags.


In [43]:
# Section 4 — Flags

import pandas as pd

df = pd.read_json("../outputs/flags/flag_success.log", lines=True, nrows=5)
df


,flag_id,user_id,rule_name,metric_value,threshold,window,event_id,event_time,ingest_time
0,aaa7cbff-d06f-4a11-9aa1-978192bbe9d9:threshold,user_03345,threshold,779.90,500,0,aaa7cbff-d06f-4a11-9aa1-978192bbe9d9,2026-03-09 12:36:24.411982+00:00,2026-03-09 12:36:24.411982+00:00
1,ebefb156-361d-4454-a00f-677cc4568873:threshold,user_00733,threshold,856.43,500,0,ebefb156-361d-4454-a00f-677cc4568873,2026-03-09 12:36:24.471947+00:00,2026-03-09 12:36:24.471947+00:00
2,4cc77952-4509-4de1-b88d-c0bce031caa4:threshold,user_09647,threshold,660.33,500,0,4cc77952-4509-4de1-b88d-c0bce031caa4,2026-03-09 12:36:24.502524+00:00,2026-03-09 12:36:24.502524+00:00
3,e7bd155f-ad1a-413f-9aa3-1ce75d359389:threshold,user_02519,threshold,813.86,500,0,e7bd155f-ad1a-413f-9aa3-1ce75d359389,2026-03-09 12:36:24.562386+00:00,2026-03-09 12:36:24.562386+00:00
4,e562fda8-f5d3-4f57-9e4b-0691bf5ae54f:threshold,user_05442,threshold,516.09,500,0,e562fda8-f5d3-4f57-9e4b-0691bf5ae54f,2026-03-09 12:36:25.062695+00:00,2026-03-09 12:36:25.062695+00:00


### Evidence

The displayed records confirm that **rule breaches are emitted as persisted flag events**.

**Proves**
- `flags.log` exists and is readable
- Flags are structured JSON records
- Flags include rule name, metric value, threshold/window, and event identifiers/timestamps

**Does NOT prove**
- That all breaches were detected
- That thresholds/windows are correct
- That outputs are deduplicated across runs

Those are addressed by policy and validation evidence.


## Section 5 — Validation Report

### Explanation

This section inspects the **validation report** to provide a consolidated,
post-run summary of system behaviour.

The claim being proven:

- Validation outcomes were recorded at run completion
- Ingestion, ordering, duplication, and contract policies were enforced
- Aggregate counts are consistent with earlier artifacts

Artifact under inspection:

- `outputs/reports/validation_report.json`

What this section **proves**:
- System-level validation results exist
- Policy enforcement outcomes are explicitly recorded

What this section **does not prove**:
- Internal implementation correctness
- That policies are sufficient for all compliance needs


In [44]:
# Section 5 — Validation Report

import json
import pandas as pd

with open("../outputs/reports/validation_success.json") as f:
    report = json.load(f)

rows = [
    {"key": "status", "value": report["status"]},
    {"key": "contract.name", "value": report["contract"]["name"]},
    {"key": "events_read", "value": report["counts"]["events_read"]},
    {"key": "events_valid", "value": report["counts"]["events_valid"]},
    {"key": "flags_emitted", "value": report["counts"]["flags_emitted"]},
    {"key": "contract_failures", "value": report["counts"]["contract_failures"]},
    {"key": "duplicate_failures", "value": report["counts"]["duplicate_failures"]},
    {"key": "ordering_failures", "value": report["counts"]["ordering_failures"]},
    {"key": "checkpoint.offset", "value": report["checkpoint"]["offset"]},
]

pd.DataFrame(rows)


,key,value
0,status,success
1,contract.name,money_flow_event_contract
2,events_read,1543
3,events_valid,1543
4,flags_emitted,387
5,contract_failures,0
6,duplicate_failures,0
7,ordering_failures,0
8,checkpoint.offset,2287784


### Evidence

The validation report confirms that the system completed successfully and
recorded explicit enforcement outcomes.

**Proves**
- Validation completed with status `success`
- Contract enforcement produced zero failures
- Ordering and duplicate policies produced zero failures
- Aggregate counts and checkpoint offset are explicitly recorded

**Does NOT prove**
- That every possible violation type was exercised
- That validation logic is complete or optimal

This concludes the evidence notebook.


## Section 6 — No-skip (Persisted Events)

### Explanation

This section proves the **no-skip guarantee within the persisted boundary** by comparing:

- the watcher’s final processed byte `offset`
- the physical size of the persisted stream file in bytes

Claim being proven:

- At watcher shutdown, `final_offset` (and checkpoint `offset`) equals the stream file size in bytes,
  implying the watcher consumed the persisted stream **through end-of-file**.

Artifacts under inspection:

- `logs/run.json` (final_offset, stream_path)
- `outputs/checkpoints/state.json` (offset)
- `data/stream.log` (file size in bytes)

Pass criteria:

- `final_offset == checkpoint.offset == stream_file_size_bytes`

Fail criteria:

- Any mismatch between these three values


In [45]:
import json
import os
import pandas as pd

with open("../logs/run_second.json") as f:
    run_log = json.load(f)

with open("../outputs/checkpoints/state_success.json") as f:
    checkpoint = json.load(f)

stream_path = "../data/sample_stream.log"
stream_size_bytes = os.path.getsize(stream_path)

rows = [
    {"metric": "stream_path", "value": stream_path},
    {"metric": "stream_file_size_bytes", "value": stream_size_bytes},
    {"metric": "run.final_offset", "value": run_log["offsets"]["final_offset"]},
    {"metric": "checkpoint.offset", "value": checkpoint["offset"]},
    {
        "metric": "no_skip_to_stream_boundary",
        "value": (
            run_log["offsets"]["final_offset"] == checkpoint["offset"] == stream_size_bytes
        ),
    },
]

pd.DataFrame(rows)

,metric,value
0,stream_path,../data/sample_stream.log
1,stream_file_size_bytes,2287784
2,run.final_offset,2287784
3,checkpoint.offset,2287784
4,no_skip_to_stream_boundary,True


### Evidence

The table shows:

- `run.final_offset == checkpoint.offset == stream_file_size_bytes`

the watcher processed the persisted stream **through end-of-file**, and no persisted bytes were skipped within this run boundary.

**Proves**
- End-of-file consumption of the persisted stream at shutdown (byte-level)

**Does NOT prove**
- That pre-persistence (in-memory) events were not lost
- Exactly-once guarantees across multiple independent watcher processes


## Section 7 — Restart-resume

### Explanation

This section proves restart–resume correctness using two consecutive run logs.

**Claim being proven**

- The first run finished processing the stream at a recorded byte offset.
- The next run resumed processing from that exact same offset.

**Artifacts under inspection**

- `logs/run_first.json`
- `logs/run_second.json`

**Pass criteria**

- run_first.offsets.final_offset == run_second.offsets.resume_offset


**Interpretation**

- The second run starts exactly where the first run ended.
- This demonstrates correct resume behavior from the persisted offset.

**What this section does NOT claim**

- Exactly-once processing guarantees  
- No duplicate flags across restarts  
- Business-rule correctness of emitted flags

In [46]:
import json
import pandas as pd

# Load run logs
with open("../logs/run_first.json") as f:
    run_first = json.load(f)

with open("../logs/run_second.json") as f:
    run_second = json.load(f)

# Extract offsets
final_offset_first = run_first["offsets"]["final_offset"]
resume_offset_second = run_second["offsets"]["resume_offset"]

rows = [
    {"metric": "run_first.final_offset", "value": final_offset_first},
    {"metric": "run_second.resume_offset", "value": resume_offset_second},
    {
        "metric": "restart_resume_match",
        "value": final_offset_first == resume_offset_second
    }
]

pd.DataFrame(rows)

,metric,value
0,run_first.final_offset,1870846
1,run_second.resume_offset,1870846
2,restart_resume_match,True


### Evidence

**PASS** if:

- `run_first.offsets.final_offset == run_second.offsets.resume_offset`

This demonstrates that the second run resumed processing exactly from the byte offset where the previous run finished.

**Does NOT prove**

- Exactly-once outputs across runs  
- No duplicated flags across restarts

## Section 8 — Fail-fast Contract (Negative Test)

### Explanation

This section proves **fail-fast behaviour** when an invalid record is present
in the persisted input stream.

An invalid line was **intentionally appended** to the existing persisted
stream file to simulate upstream corruption or malformed input.

Claim being proven:

- The watcher fails immediately on encountering an invalid record
- The run is marked as failed
- No further processing or flag emission continues after the failure

Artifact under inspection:

- `logs/run_bad.json`
- `outputs/reports/validation_bad.json`
- `outputs/flags/flags_bad.log` (if created)

This negative test is **isolated** and does not affect other sections.


In [47]:
# Section 8 — Fail-fast Contract

import json
import pandas as pd
from pathlib import Path

with open("../logs/run_fail.json") as f:
    run_bad = json.load(f)

with open("../outputs/reports/validation_fail.json") as f:
    validation_bad = json.load(f)

flags_bad_path = Path("../outputs/flags/flags_bad.log")
flags_bad_lines = 0
if flags_bad_path.exists():
    with open(flags_bad_path) as f:
        flags_bad_lines = sum(1 for _ in f)

rows = [
    {"metric": "run.status", "value": run_bad.get("status")},
    {"metric": "run.events_read", "value": run_bad.get("counts", {}).get("events_read")},
    {"metric": "run.flags_emitted", "value": run_bad.get("counts", {}).get("flags_emitted")},
    {"metric": "validation.status", "value": validation_bad.get("status")},
    {"metric": "contract_failures", "value": validation_bad.get("counts", {}).get("contract_failures")},
    {"metric": "json_parse_failures", "value": validation_bad.get("counts", {}).get("json_parse_failures")},
    {"metric": "flags_file_exists", "value": flags_bad_path.exists()},
    {"metric": "flags_line_count", "value": flags_bad_lines},
]

pd.DataFrame(rows)


,metric,value
0,run.status,failed
1,run.events_read,1
2,run.flags_emitted,0
3,validation.status,failed
4,contract_failures,0
5,json_parse_failures,1
6,flags_file_exists,True
7,flags_line_count,0


### Evidence

The validation report shows that the watcher failed immediately upon encountering
an invalid record.

Observed:

- A JSON parse failure was recorded (`json_parse_failures = 1`)
- Validation status is `failed`
- Flags were not emitted

**PASS condition**

- The run fails on the first invalid record
- No recovery, coercion, or continued processing occurs after the failure

**Clarification**

- Failure occurs at first invalid record; no recovery/coercion is applied.
- The system does not guarantee “zero outputs” for a run containing invalid data.

This confirms **fail-fast behaviour at the point of contract violation**.
